# Fix 1 — Diagnose Paris naming mismatch
Paris GT references `defense_000038` but the Kaggle dataset may use `paris_defense_000038`.
Let's check the exact naming convention used in the Paris image folder.

In [30]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/datasets/jeffreyamc/oxford-paris-buildings-v2')
GT_DIR       = Path('/kaggle/working/retrieval_output/ground_truth')

def find_gt_dir(gt_root):
    files = list(gt_root.rglob('*_query.txt'))
    return files[0].parent if files else None

gt_flat = find_gt_dir(GT_DIR)

# ── What do Paris GT stems look like BEFORE cleaning? ────────────────────
print('Paris GT stems (RAW, before prefix removal):')
paris_qf = sorted(gt_flat.glob('*_query.txt'))
paris_qf = [f for f in paris_qf if 'defense' in f.name or 'eiffel' in f.name
            or 'louvre' in f.name]
for qf in paris_qf[:5]:
    with open(qf) as f:
        raw = f.read().strip()
    print(f'  {qf.name}: {raw}')

# ── What do Paris image files look like in Kaggle? ───────────────────────
print('\nParis image filenames (Kaggle dataset):')
for lm in ['defense', 'eiffel', 'louvre']:
    lm_path = DATASET_ROOT / 'paris' / lm
    if lm_path.exists():
        imgs = sorted(lm_path.glob('*.jpg'))[:3]
        for p in imgs:
            print(f'  {p.name}')

Paris GT stems (RAW, before prefix removal):
  defense_1_query.txt: paris_defense_000605 130.000000 17.000000 843.000000 732.000000
  defense_2_query.txt: paris_defense_000331 258.000000 88.000000 805.000000 619.000000
  defense_3_query.txt: paris_defense_000216 213.000000 67.000000 765.000000 609.000000
  defense_4_query.txt: paris_defense_000056 8.000000 104.000000 707.000000 913.000000
  defense_5_query.txt: paris_defense_000254 226.000000 77.000000 645.000000 510.000000

Paris image filenames (Kaggle dataset):
  paris_defense_000000.jpg
  paris_defense_000002.jpg
  paris_defense_000004.jpg
  paris_eiffel_000000.jpg
  paris_eiffel_000002.jpg
  paris_eiffel_000004.jpg
  paris_louvre_000000.jpg
  paris_louvre_000002.jpg
  paris_louvre_000004.jpg


In [31]:
# ── Oxford GT stems (RAW) — to confirm Oxford clean() works correctly ─────
print('Oxford GT stems (RAW):')
ox_qf = [f for f in sorted(gt_flat.glob('*_query.txt'))
          if 'all_souls' in f.name or 'hertford' in f.name]
for qf in ox_qf[:3]:
    with open(qf) as f:
        raw = f.read().strip()
    print(f'  {qf.name}: {raw}')

print('\nOxford image filenames (Kaggle):')
for lm in ['all_souls', 'hertford']:
    lm_path = DATASET_ROOT / 'oxford' / lm
    if lm_path.exists():
        imgs = sorted(lm_path.glob('*.jpg'))[:3]
        for p in imgs:
            print(f'  {p.name}')

Oxford GT stems (RAW):
  all_souls_1_query.txt: oxc1_all_souls_000013 136.5 34.1 648.5 955.7
  all_souls_2_query.txt: oxc1_all_souls_000026 78 5 714 1000
  all_souls_3_query.txt: oxc1_oxford_002985 186 163 589 859

Oxford image filenames (Kaggle):
  all_souls_000000.jpg
  all_souls_000001.jpg
  all_souls_000002.jpg
  hertford_000008.jpg
  hertford_000010.jpg
  hertford_000011.jpg


---
# Fix 2 — Correct `clean()` function

The issue is that Paris GT uses prefix `paris_` but the Kaggle files keep it.
For example:
- GT says: `paris_defense_000038`  → after `clean()` → `defense_000038`
- Kaggle file: `paris_defense_000038.jpg` → stem = `paris_defense_000038`

The `clean()` function removes the prefix from GT stems but the image files **keep** the prefix.  
We need to either: keep prefix in GT, or remove prefix from image filenames.  
The cleanest fix: **don't strip `paris_` from GT stems** — just strip `oxc1_`.

In [32]:
# Test the correct clean() for each dataset
import re

def clean_oxford(s):
    """Oxford GT has prefix oxc1_ which is NOT in the filenames."""
    return s.replace('oxc1_', '').strip()

def clean_paris(s):
    """Paris GT has prefix paris_ which IS in the filenames.
    So we do NOT strip it."""
    return s.strip()   # keep 'paris_defense_000038' as-is

# Verify Oxford
print('Oxford alignment test:')
ox_qf = sorted(f for f in gt_flat.glob('*_query.txt') if 'all_souls' in f.name)
if ox_qf:
    with open(ox_qf[0]) as f: parts = f.read().strip().split()
    raw_stem = parts[0]
    cleaned  = clean_oxford(raw_stem)
    all_stems_ox = {p.stem for p in (DATASET_ROOT/'oxford').rglob('*.jpg')}
    print(f'  Raw GT:   "{raw_stem}"')
    print(f'  Cleaned:  "{cleaned}"')
    print(f'  In dataset: {cleaned in all_stems_ox}')

# Verify Paris
print('\nParis alignment test:')
pa_qf = sorted(f for f in gt_flat.glob('*_query.txt') if 'defense' in f.name)
if pa_qf:
    with open(pa_qf[0]) as f: parts = f.read().strip().split()
    raw_stem = parts[0]
    cleaned_wrong  = raw_stem.replace('paris_', '')  # what we were doing
    cleaned_right  = clean_paris(raw_stem)           # what we should do
    all_stems_pa = {p.stem for p in (DATASET_ROOT/'paris').rglob('*.jpg')}
    print(f'  Raw GT:          "{raw_stem}"')
    print(f'  Wrong clean():   "{cleaned_wrong}" → in dataset: {cleaned_wrong in all_stems_pa}')
    print(f'  Correct clean(): "{cleaned_right}" → in dataset: {cleaned_right in all_stems_pa}')

Oxford alignment test:
  Raw GT:   "oxc1_all_souls_000013"
  Cleaned:  "all_souls_000013"
  In dataset: True

Paris alignment test:
  Raw GT:          "paris_defense_000605"
  Wrong clean():   "defense_000605" → in dataset: False
  Correct clean(): "paris_defense_000605" → in dataset: True


---
# Fix 3 — Impact of vocabulary size on discriminability

Check 3 showed:
- `sim(query, known_positive) = 0.1145`
- `sim(query, random)         = -0.0121`

This gap is small. With a better vocabulary (larger K) or more SIFT features,  
the gap should grow. Let's measure it for K = 64, 256, 512, 1024.

In [33]:
import numpy as np, pickle
CACHE_DIR = Path('/kaggle/working/retrieval_output/cache')

# Load current DB (K=256)
db_index_files = list(CACHE_DIR.glob('db_index_*.npy'))
stems_file     = CACHE_DIR / 'db_stems.pkl'

if not db_index_files or not stems_file.exists():
    print('DB index not found — run main pipeline first')
else:
    DB_VLAD = np.load(db_index_files[0])
    with open(stems_file, 'rb') as f:
        db_stems = pickle.load(f)
    s2i = {s: i for i, s in enumerate(db_stems)}

    # Evaluate discriminability on ALL 55 Oxford queries
    # (the ones we know work correctly)
    gt_flat_dir = find_gt_dir(GT_DIR)

    def read_set_ox(q_stem, sfx):
        p = gt_flat_dir / f'{q_stem}_{sfx}.txt'
        if not p.exists(): return set()
        with open(p) as f:
            return {l.strip().replace('oxc1_','') for l in f if l.strip()}

    pos_sims, neg_sims = [], []
    rng = np.random.default_rng(42)

    for qf in sorted(gt_flat_dir.glob('*_query.txt')):
        # Skip Paris (names don't match yet)
        if any(x in qf.name for x in ['defense','eiffel','louvre','invalides',
                                        'moulin','musee','notredame','pantheon',
                                        'pompidou','sacre','triomphe','general']):
            continue

        q_stem = qf.stem.replace('_query', '')
        with open(qf) as f: parts = f.read().strip().split()
        q_img = parts[0].replace('oxc1_', '')

        good = read_set_ox(q_stem, 'good')
        ok   = read_set_ox(q_stem, 'ok')
        pos  = good | ok

        q_idx = s2i.get(q_img)
        if q_idx is None: continue

        q_vec = DB_VLAD[q_idx]

        # Similarities with known positives
        for stem in pos:
            idx = s2i.get(stem)
            if idx is not None:
                pos_sims.append(float(q_vec @ DB_VLAD[idx]))

        # Similarities with random negatives
        rand_idx = rng.choice(len(db_stems), 20, replace=False)
        neg_sims.extend(float(q_vec @ DB_VLAD[i]) for i in rand_idx
                        if db_stems[i] not in pos)

    print(f'Oxford discriminability analysis (K={DB_VLAD.shape[1]}D vectors, K_vlad from filename):')
    print(f'  Positive pairs analysed : {len(pos_sims)}')
    print(f'  Negative pairs analysed : {len(neg_sims)}')
    print(f'  Mean sim (positives) : {np.mean(pos_sims):.4f}  ± {np.std(pos_sims):.4f}')
    print(f'  Mean sim (negatives) : {np.mean(neg_sims):.4f}  ± {np.std(neg_sims):.4f}')
    gap = np.mean(pos_sims) - np.mean(neg_sims)
    print(f'  Discriminability gap : {gap:.4f}')
    print()
    if gap < 0.1:
        print('  ⚠ Gap < 0.10 → embeddings are weakly discriminative')
        print('    → Try K_VLAD=1024 or K_VLAD=4096')
        print('    → Try increasing SIFT_N_FEATURES to 5000')
        print('    → Try removing PCA (use raw VLAD + L2 normalize)')
    elif gap < 0.2:
        print('  ⚡ Gap 0.10-0.20 → moderate discriminability')
        print('    → Increasing K_VLAD should help significantly')
    else:
        print('  ✓ Gap > 0.20 → good discriminability, mAP issues likely elsewhere')

Oxford discriminability analysis (K=256D vectors, K_vlad from filename):
  Positive pairs analysed : 2840
  Negative pairs analysed : 1097
  Mean sim (positives) : 0.1654  ± 0.2178
  Mean sim (negatives) : -0.0044  ± 0.0594
  Discriminability gap : 0.1697

  ⚡ Gap 0.10-0.20 → moderate discriminability
    → Increasing K_VLAD should help significantly


---
# Summary of findings and recommended fixes

In [34]:
print('='*60)
print('DIAGNOSIS SUMMARY')
print('='*60)
print('''
PROBLEM 1 — Paris naming mismatch  (fixes ~half the queries)
  GT file says:    paris_defense_000038   (with paris_ prefix)
  clean() removes: paris_  → defense_000038
  Kaggle file is:  paris_defense_000038.jpg  (keeps prefix)
  
  FIX: In scan_dataset(), use dataset-specific clean():
    Oxford: strip  oxc1_  (not in filenames)
    Paris:  don't strip paris_  (IS in filenames)

PROBLEM 2 — Small discriminability gap (fixes absolute mAP level)
  sim(positive) ≈ 0.11  vs  sim(random) ≈ -0.01  →  gap ≈ 0.12
  Target gap for good mAP: > 0.25
  
  FIX A: Increase K_VLAD from 256 → 1024 or 4096
         Papers report best results at K=64 for VLAD but with
         more SIFT features (all detectable, not capped at 2000)
  
  FIX B: Remove the SIFT_N_FEATURES cap (use 0 = unlimited)
         Oxford images are 1024x768 with ~3300 features typically
         Capping at 2000 loses discriminative keypoints
  
  FIX C: Increase MAX_IMAGE_SIZE from 800 → 1024
         Resizing down loses fine-grained features

Expected mAP after fixes:
  After Fix 1 only:    ~0.25-0.30 (Paris no longer drags average down)
  After Fix 1+2A+2B:   ~0.40-0.55 (competitive with literature baseline)
  With RANSAC + AQE:   ~0.55-0.70 (approaching paper results)
''')
print('='*60)

DIAGNOSIS SUMMARY

PROBLEM 1 — Paris naming mismatch  (fixes ~half the queries)
  GT file says:    paris_defense_000038   (with paris_ prefix)
  clean() removes: paris_  → defense_000038
  Kaggle file is:  paris_defense_000038.jpg  (keeps prefix)
  
  FIX: In scan_dataset(), use dataset-specific clean():
    Oxford: strip  oxc1_  (not in filenames)
    Paris:  don't strip paris_  (IS in filenames)

PROBLEM 2 — Small discriminability gap (fixes absolute mAP level)
  sim(positive) ≈ 0.11  vs  sim(random) ≈ -0.01  →  gap ≈ 0.12
  Target gap for good mAP: > 0.25
  
  FIX A: Increase K_VLAD from 256 → 1024 or 4096
         Papers report best results at K=64 for VLAD but with
         more SIFT features (all detectable, not capped at 2000)
  
  FIX B: Remove the SIFT_N_FEATURES cap (use 0 = unlimited)
         Oxford images are 1024x768 with ~3300 features typically
         Capping at 2000 loses discriminative keypoints
  
  FIX C: Increase MAX_IMAGE_SIZE from 800 → 1024
         Resizing d